## 1. Load data

In [4]:
import pandas as pd

reviews_df = pd.read_csv('IMDB.csv')
reviews_df.head()

,review,sentiment
0,Film version of Sandra Bernhard's one-woman of...,negative
1,I switched this on (from cable) on a whim and ...,positive
2,The `plot' of this film contains a few holes y...,negative
3,"Some amusing humor, some that falls flat, some...",negative
4,What can you say about this movie? It was not ...,negative


## 2. Split the data into train and test sets

In [5]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(reviews_df, test_size=0.2, random_state=42, stratify=reviews_df['sentiment'])

## 3. Convert the reviews into vectors (embeddings)

In [2]:
import os
from transformers import AutoTokenizer
from optimum.onnxruntime import ORTModelForFeatureExtraction

model_name = 'sentence-transformers/all-mpnet-base-v2'
save_folder = 'production_onnx_model/'

os.makedirs(save_folder, exist_ok=True)

print(f"Translating {model_name} to ONNX...")

# It downloads the PyTorch model and immediately translates the math into ONNX format.
onnx_model = ORTModelForFeatureExtraction.from_pretrained(model_name, export=True)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 3. Save it to your hard drive permanently
onnx_model.save_pretrained(save_folder)
tokenizer.save_pretrained(save_folder)

print(f"✅ Success! Your ONNX model is saved in the '{save_folder}' folder.")

Translating sentence-transformers/all-mpnet-base-v2 to ONNX... (This takes about 1-2 minutes)


The model sentence-transformers/all-mpnet-base-v2 was already converted to ONNX but got `export=True`, the model will be converted to ONNX once again. Don't forget to save the resulting model with `.save_pretrained()`
`torch_dtype` is deprecated! Use `dtype` instead!


✅ Success! Your ONNX model is saved in the 'production_onnx_model/' folder.


In [ ]:
import numpy as np
import torch
from tqdm.auto import tqdm
from optimum.onnxruntime import ORTModelForFeatureExtraction
from transformers import AutoTokenizer

# 1. Load your saved ONNX model and Tokenizer
model_path = "production_onnx_model/"
tokenizer = AutoTokenizer.from_pretrained(model_path)
onnx_model = ORTModelForFeatureExtraction.from_pretrained(model_path)

# 2. Define the production batching function
def encode_series_in_batches(text_series, batch_size=32):
    """
    Safely converts a Pandas Series of text into an array of embeddings.
    """
    # Convert the Pandas Series to a standard Python list (much faster to slice)
    texts = text_series.tolist()
    all_embeddings = []
    
    # Loop through the list in chunks (batches)
    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding Text"):
        batch_texts = texts[i : i + batch_size]
        
        # 1. Tokenize the current batch
        inputs = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt")
        
        # 2. Run the ONNX model
        outputs = onnx_model(**inputs)
        
        # 3. Perform Mean Pooling
        token_embeddings = outputs.last_hidden_state
        attention_mask = inputs['attention_mask'].unsqueeze(-1).expand(token_embeddings.size()).float()
        
        sum_embeddings = torch.sum(token_embeddings * attention_mask, 1)
        sum_mask = torch.clamp(attention_mask.sum(1), min=1e-9)
        
        batch_embeddings = (sum_embeddings / sum_mask).detach().numpy()
        
        # Store the results of this batch
        all_embeddings.append(batch_embeddings)
        
    # Stack all the tiny batch arrays back into one massive matrix
    return np.vstack(all_embeddings)

In [8]:
train_embeddings = encode_series_in_batches(train_df['review'], batch_size=32)
test_embeddings = encode_series_in_batches(test_df['review'], batch_size=32)

Encoding Text: 100%|██████████| 7/7 [01:22<00:00, 11.83s/it]


In [9]:
train_embeddings.shape, test_embeddings.shape

((800, 768), (200, 768))

In [10]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
from lightgbm import LGBMClassifier


X_train = train_embeddings
y_train = train_df['sentiment'].map({'positive': 1, 'negative': 0}).values
X_test = test_embeddings
y_test = test_df['sentiment'].map({'positive': 1, 'negative': 0}).values

models_list = [LogisticRegression(max_iter=1000, random_state=42), 
               SGDClassifier(max_iter=1000, random_state=42),
               LinearSVC(random_state=42, max_iter=1000),
               SVC(random_state=42, max_iter=1000, kernel='rbf'),
               MLPClassifier(random_state=42, max_iter=1000),
               LGBMClassifier(random_state=42, max_iter=1000, n_jobs=-1, verbose=-1)]

for model in models_list:
  model.fit(X_train, y_train)
  score = model.score(X_test, y_test)
  predictions = model.predict(X_test)
  precision = precision_score(y_test, predictions)
  recall = recall_score(y_test, predictions)
  f1 = f1_score(y_test, predictions)
  print(f"{model.__class__.__name__} : Accuracy = {score:.4f}, Precision = {precision:.4f}, Recall = {recall:.4f}, F1 = {f1:.4f}")


LogisticRegression : Accuracy = 0.7950, Precision = 0.7745, Recall = 0.8144, F1 = 0.7940
SGDClassifier : Accuracy = 0.7950, Precision = 0.7857, Recall = 0.7938, F1 = 0.7897
LinearSVC : Accuracy = 0.8100, Precision = 0.7810, Recall = 0.8454, F1 = 0.8119
SVC : Accuracy = 0.8150, Precision = 0.8061, Recall = 0.8144, F1 = 0.8103
MLPClassifier : Accuracy = 0.8050, Precision = 0.7788, Recall = 0.8351, F1 = 0.8060
LGBMClassifier : Accuracy = 0.7800, Precision = 0.7732, Recall = 0.7732, F1 = 0.7732


c:\Users\Nirma\Desktop\ml projects\movie-review-sentiment-analysis\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Nirma\Desktop\ml projects\movie-review-sentiment-analysis\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [29]:
from scipy.stats import loguniform
from sklearn.model_selection import RandomizedSearchCV
from sklearn.neural_network import MLPClassifier


mlp_param_dist = {
    # Network architecture: single layer vs. two layers
    'hidden_layer_sizes': [(100,), (50, 50), (128, 64)],
    # Activation function for hidden layers
    'activation': ['relu', 'tanh'],
    # L2 penalty (regularization term) parameter
    'alpha': loguniform(1e-4, 1e-1),
    # Initial learning rate
    'learning_rate_init': [0.001, 0.01, 0.05],
    # Solver for weight optimization
    'solver': ['adam', 'sgd']
}


random_search_mlp = RandomizedSearchCV(MLPClassifier(max_iter=2000), mlp_param_dist, n_iter=100, cv=3, verbose=2, random_state=42, n_jobs=-1, error_score='raise')
random_search_mlp.fit(train_embeddings, train_df['sentiment'].map({'positive': 1, 'negative': 0}).values)

print(f"Best parameters: {random_search_mlp.best_params_}")
print(f"Best cross-validation score: {random_search_mlp.best_score_:.4f}")


Fitting 3 folds for each of 100 candidates, totalling 300 fits
Best parameters: {'activation': 'tanh', 'alpha': np.float64(0.003600575029200904), 'hidden_layer_sizes': (100,), 'learning_rate_init': 0.001, 'solver': 'sgd'}
Best cross-validation score: 0.8387


In [30]:
from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV
from lightgbm import LGBMClassifier

lgbm_param_dist = {
    # Number of boosted trees to fit
    'n_estimators': randint(100, 500),
    # Maximum tree leaves for base learners
    'num_leaves': randint(20, 100),
    # Maximum tree depth (-1 means no limit)
    'max_depth': [-1, 10, 20, 30],
    # Boosting learning rate
    'learning_rate': loguniform(1e-3, 1e-1),
    # Fraction of features selected randomly for each tree
    'colsample_bytree': uniform(0.6, 0.4),
    # Fraction of data to use for each tree (prevents overfitting)
    'subsample': uniform(0.6, 0.4)
}


random_search_lgbm = RandomizedSearchCV(LGBMClassifier(), lgbm_param_dist, n_iter=100, cv=3, verbose=2, random_state=42, n_jobs=-1, error_score='raise')
random_search_lgbm.fit(train_embeddings, train_df['sentiment'].map({'positive': 1, 'negative': 0}).values)

print(f"Best parameters: {random_search_lgbm.best_params_}")
print(f"Best cross-validation score: {random_search_lgbm.best_score_:.4f}")

Fitting 3 folds for each of 100 candidates, totalling 300 fits
Best parameters: {'colsample_bytree': np.float64(0.6592347719813599), 'learning_rate': np.float64(0.0989648498531856), 'max_depth': 20, 'n_estimators': 212, 'num_leaves': 21, 'subsample': np.float64(0.8010716092915446)}
Best cross-validation score: 0.8437


In [31]:
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV


svc_param_dist = {
    # Regularization parameter
    'C': loguniform(1e-1, 1e2),
    # Kernel coefficient
    'gamma': ['scale', 'auto', 0.01, 0.1, 1],
    # Class weights (helpful if your positive/negative reviews are imbalanced)
    'class_weight': [None, 'balanced']
}


random_search_svc = RandomizedSearchCV(SVC(), svc_param_dist, n_iter=100, cv=3, verbose=2, random_state=42, n_jobs=-1, error_score='raise')
random_search_svc.fit(train_embeddings, train_df['sentiment'].map({'positive': 1, 'negative': 0}).values)

print(f"Best parameters: {random_search_svc.best_params_}")
print(f"Best cross-validation score: {random_search_svc.best_score_:.4f}")

Fitting 3 folds for each of 100 candidates, totalling 300 fits
Best parameters: {'C': np.float64(0.5326964650459663), 'class_weight': 'balanced', 'gamma': 1}
Best cross-validation score: 0.8462


In [32]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LogisticRegression


param_distributions = [
    # Search Space 1: For 'liblinear' (Supports L1 and L2)
    {
        'solver': ['liblinear'],
        'penalty': ['l1', 'l2'],
        'C': [0.01, 0.1, 1, 10],
        'max_iter': [500, 1000]
    },
    # Search Space 2: For 'saga' (Supports ElasticNet and l1_ratio)
    {
        'solver': ['saga'],
        'penalty': ['elasticnet'],
        'l1_ratio': [0, 0.5, 1], # Now it is safe to use!
        'C': [0.01, 0.1, 1, 10],
        'max_iter': [500, 1000]
    }
]

random_search_lr = RandomizedSearchCV(LogisticRegression(), param_distributions, n_iter=100, cv=3, verbose=2, random_state=42, n_jobs=-1, error_score='raise')
random_search_lr.fit(train_embeddings, train_df['sentiment'].map({'positive': 1, 'negative': 0}).values)

print(f"Best parameters: {random_search_lr.best_params_}")
print(f"Best cross-validation score: {random_search_lr.best_score_:.4f}")

Fitting 3 folds for each of 40 candidates, totalling 120 fits


c:\Users\Nirma\Desktop\ml projects\movie-review-sentiment-analysis\.venv\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 40 is smaller than n_iter=100. Running 40 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best parameters: {'solver': 'liblinear', 'penalty': 'l2', 'max_iter': 500, 'C': 10}
Best cross-validation score: 0.8362


c:\Users\Nirma\Desktop\ml projects\movie-review-sentiment-analysis\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [33]:
models = [random_search_lr.best_estimator_, random_search_mlp.best_estimator_, random_search_lgbm.best_estimator_, random_search_svc.best_estimator_]
model_names = ['Logistic Regression', 'MLPClassifier', 'LGBMClassifier', 'SVC']

In [34]:
from sklearn.metrics import accuracy_score, classification_report

for model, name in zip(models, model_names):
    print(f"\nEvaluating {name}...")
    model_predictions = model.predict(test_embeddings)
    accuracy = accuracy_score(test_df['sentiment'].map({'positive': 1, 'negative': 0}).values, model_predictions)
    report = classification_report(test_df['sentiment'].map({'positive': 1, 'negative': 0}).values, model_predictions, target_names=['negative', 'positive'])
    print(f"Test Accuracy: {accuracy:.4f}")
    print("Classification Report:")
    print(report)


Evaluating Logistic Regression...
Test Accuracy: 0.7850
Classification Report:
              precision    recall  f1-score   support

    negative       0.81      0.76      0.78       103
    positive       0.76      0.81      0.79        97

    accuracy                           0.79       200
   macro avg       0.79      0.79      0.78       200
weighted avg       0.79      0.79      0.78       200


Evaluating MLPClassifier...
Test Accuracy: 0.8200
Classification Report:
              precision    recall  f1-score   support

    negative       0.82      0.83      0.83       103
    positive       0.82      0.80      0.81        97

    accuracy                           0.82       200
   macro avg       0.82      0.82      0.82       200
weighted avg       0.82      0.82      0.82       200


Evaluating LGBMClassifier...
Test Accuracy: 0.7750
Classification Report:
              precision    recall  f1-score   support

    negative       0.77      0.80      0.78       103
    posi

c:\Users\Nirma\Desktop\ml projects\movie-review-sentiment-analysis\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [39]:
input_reviews = ["On paper, this film should have been the masterpiece of the decade. The cinematography is absolutely breathtaking, capturing the sweeping vistas with stunning clarity, and the wardrobe department deserves a medal. It’s truly impressive how much money and effort went into making sure every single visual detail was flawless, presumably to distract the audience from the agonizingly slow two hours of absolutely nothing happening. I am genuinely amazed that a movie with such a brilliant cast and spectacular visuals could manage to be this painfully boring. An unforgettable cure for insomnia.",
                 "I really wanted to hate this movie. The trailers made it look like a cheap, derivative cash-grab with a completely absurd premise. And honestly, the first twenty minutes are a total disaster, filled with clunky dialogue, awful pacing, and confusing camera work. But then, something inexplicably magical happens. The utter ridiculousness of the plot wraps around to become pure comedic genius. The actors lean into the terrible script with such phenomenal charisma that you can't help but be charmed. It is an absolute trainwreck of a film, but it is undeniably the most fun I've had at the theater all year. A glorious disaster.",
                 "This movie is a complete disaster, and I couldn't be happier. The director clearly had no idea what they were doing, resulting in the most hilariously awkward dialogue and absurd plot twists I've ever witnessed. It's garbage, absolute garbage, but it is the most entertaining piece of trash I have watched all decade. Five stars, I will be forcing all my friends to watch this.",
                 "With an all-star cast, an incredible visionary director, and a budget of 200 million dollars, this film promised to be the ultimate cinematic experience. The special effects were top-notch, and the score was composed beautifully. What a spectacular way to deliver an entirely soulless, empty, and painfully tedious story. I am in awe of how they managed to make something so pretty feel so incredibly hollow. A monumental waste of talent.",
                 "I walked into the theater expecting absolutely nothing. The poster was ugly, the marketing was annoying, and the lead actor usually gets on my last nerve. Imagine my shock when I found myself laughing out loud within the first five minutes. It completely subverted my expectations, delivering a smart, witty, and surprisingly emotional journey that kept me glued to the screen. I happily stand corrected."]
y_true = [0, 1, 1, 0, 1]

input_embeddings = embedder.encode(input_reviews, show_progress_bar=False)
best_model = random_search_mlp.best_estimator_
predictions = best_model.predict(input_embeddings)
print(predictions)

[0 0 0 0 1]


### **conclusion :** most of the edge cases are classified correctly, since we are using sentence embedding instead of word embeddings we are able to keep the semaintic meaning of the text.